# VeriClue — fine-tune detector vùng CCCD trên Google Colab

**Notebook này train vùng thông tin, chưa train VietOCR đọc chữ.**
Faster R-CNN + MobileNetV3 pretrained COCO, input 640 px, 10 lớp trường CCCD.
YOLO chỉ là định dạng nhãn nguồn; không cài Ultralytics/YOLOv5.

Runtime → Change runtime type → **T4 GPU**, rồi Run all với `SMOKE=True`.
Smoke chỉ kiểm tra training chạy, không là benchmark. AUTO_GROUP=True tự đọc số CCCD
để gom nhóm tạm, loại ảnh không chắc. Kết quả chỉ là thử nghiệm trên tập con dễ đọc,
chưa chứng minh chia tập theo người. AUTO_GROUP=False dùng groups.csv đã duyệt thủ công.
Test giữ riêng, không dùng chọn checkpoint/ngưỡng. Không có accuracy OCR nếu chưa có transcript.
Ảnh, crop và checkpoint nằm trong Drive riêng của bạn; không chia sẻ công khai notebook có output chứa CCCD.

Dataset nguồn: https://www.kaggle.com/datasets/tranphucthaitruong/vietnamese-card-dataset (v1).
Metadata Kaggle ghi Unknown, README ghi CC BY 4.0: quyền nguồn còn cần xác minh trước phát hành.

In [ ]:
%pip -q install pyyaml==6.0.3 easyocr==1.7.2
from pathlib import Path

import torch
import torchvision
from google.colab import drive

drive.mount("/content/drive")
print("torch", torch.__version__, "torchvision", torchvision.__version__)
assert torch.cuda.is_available(), "Bật T4 GPU trong Runtime > Change runtime type"
print(torch.cuda.get_device_name(0))

In [ ]:
# AUTO_GROUP tự gom theo OCR số CCCD; chưa phải nhóm danh tính đã xác minh.
SMOKE = True
AUTO_GROUP = True
EPOCHS = 15
BATCH_SIZE = 2  # Nếu hết VRAM: dùng 1 và RUN_NAME mới.
RESUME = True  # Tiếp tục từ epoch hoàn thành gần nhất.
RUN_NAME = "smoke_v1" if SMOKE else ("detector_auto_v1" if AUTO_GROUP else "detector_v1")
BASE = Path("/content/drive/MyDrive/vericlue_ocr")
BASE.mkdir(parents=True, exist_ok=True)
ROOT = Path("/content/vietnamese_card_v1")
AUDIT = BASE / "audit_v1"
RUN = BASE / "runs" / RUN_NAME
GROUPS = AUDIT / "groups.csv"

## Nạp code độc lập — không cần clone repo hoặc token GitHub

In [ ]:
Path("/content/card_dataset.py").write_text(
    '"""Audit YOLO card annotations and prepare a detector experiment (offline)."""\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport yaml\nfrom PIL import Image\n\nFIELDS = [\n    "id",\n    "name",\n    "dob",\n    "gender",\n    "nationality",\n    "expire_date",\n    "origin_place1",\n    "origin_place2",\n    "current_place1",\n    "current_place2",\n]\n\n\ndef sha256(path):\n    with Path(path).open("rb") as stream:\n        return (\n            hashlib.file_digest(stream, "sha256").hexdigest()\n            if hasattr(hashlib, "file_digest")\n            else hashlib.sha256(stream.read()).hexdigest()\n        )\n\n\ndef parse_labels(text, names, width, height):\n    """Validate normalized YOLO boxes; tolerate only rounding at image edges."""\n    boxes = []\n    for line in text.splitlines():\n        if not line.strip():\n            continue\n        parts = line.split()\n        category = int(parts[0])\n        values = list(map(float, parts[1:]))\n        if not all(math.isfinite(v) for v in values):\n            raise ValueError("Nonfinite coordinates")\n        polygon = None\n        if len(values) == 4:\n            x, y, w, h = values\n        elif len(values) >= 6 and len(values) % 2 == 0:\n            if min(values) < 0 or max(values) > 1:\n                raise ValueError("Polygon outside image")\n            xs, ys = values[::2], values[1::2]\n            x, y = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2\n            w, h = max(xs) - min(xs), max(ys) - min(ys)\n            polygon = [[px * width, py * height] for px, py in zip(xs, ys, strict=True)]\n        else:\n            raise ValueError("Expected YOLO box or polygon")\n        if not 0 <= category < len(names) or not all(math.isfinite(v) for v in (x, y, w, h)):\n            raise ValueError("Invalid class or nonfinite coordinates")\n        if (\n            w <= 0\n            or h <= 0\n            or min(x - w / 2, y - h / 2) < -1e-6\n            or max(x + w / 2, y + h / 2) > 1 + 1e-6\n        ):\n            raise ValueError("Box outside image or empty")\n        boxes.append(\n            {\n                "field": names[category],\n                "polygon": polygon,\n                "xyxy": [\n                    max(0, (x - w / 2) * width),\n                    max(0, (y - h / 2) * height),\n                    min(width, (x + w / 2) * width),\n                    min(height, (y + h / 2) * height),\n                ],\n            }\n        )\n    return boxes\n\n\ndef audit(root, out):\n    """Write reproducible manifests; never infer identities from filenames."""\n    root, out = Path(root).resolve(), Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    config = yaml.safe_load((root / "data.yaml").read_text(encoding="utf-8"))\n    names = config["names"]\n    if isinstance(names, dict):\n        names = [names[i] for i in range(len(names))]\n    rows, errors = [], []\n    pixel_groups, source_groups = defaultdict(list), defaultdict(list)\n    counts, sizes = Counter(), Counter()\n    for split in ("train", "valid", "test"):\n        images = sorted((root / split / "images").glob("*"))\n        image_stems = {p.stem for p in images}\n        for label in sorted((root / split / "labels").glob("*.txt")):\n            if label.stem not in image_stems:\n                errors.append(\n                    {"file": label.relative_to(root).as_posix(), "reason": "orphan_label"}\n                )\n        for path in images:\n            relative = path.relative_to(root).as_posix()\n            label = root / split / "labels" / (path.stem + ".txt")\n            try:\n                with Image.open(path) as image:\n                    image = image.convert("RGB")\n                    width, height = image.size\n                    pixel_hash = hashlib.sha256(\n                        str(image.size).encode() + image.tobytes()\n                    ).hexdigest()\n                boxes = parse_labels(label.read_text(encoding="utf-8"), names, width, height)\n            except (OSError, ValueError) as exc:\n                errors.append({"file": relative, "reason": str(exc)})\n                continue\n            counts.update(b["field"] for b in boxes)\n            sizes[f"{width}x{height}"] += 1\n            row = {\n                "image": relative,\n                "split": split,\n                "sha256": sha256(path),\n                "label_sha256": sha256(label),\n                "pixel_sha256": pixel_hash,\n                "width": width,\n                "height": height,\n                "boxes": boxes,\n                "cccd_candidate": any(b["field"] == "cccd" for b in boxes),\n            }\n            rows.append(row)\n            pixel_groups[pixel_hash].append(relative)\n            source_groups[path.stem.split(".rf.")[0]].append(relative)\n\n    def cross(groups):\n        return [v for v in groups.values() if len({p.split("/")[0] for p in v}) > 1]\n\n    report = {\n        "images": len(rows),\n        "splits": dict(Counter(r["split"] for r in rows)),\n        "cccd_candidates": dict(Counter(r["split"] for r in rows if r["cccd_candidate"])),\n        "classes": names,\n        "box_counts": dict(counts),\n        "sizes": dict(sizes),\n        "polygon_boxes": sum(b["polygon"] is not None for r in rows for b in r["boxes"]),\n        "errors": errors,\n        "cross_split_identical_pixels": cross(pixel_groups),\n        "cross_split_source_name_candidates": cross(source_groups),\n        "person_split_verified": False,\n        "note": "Filename and pixel checks cannot rule out same-person leakage. CCCD class is a selection hint, not verified card type.",\n    }\n    (out / "audit.json").write_text(\n        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    (out / "manifest.json").write_text(\n        json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    # Do not overwrite human-reviewed group IDs on a rerun.\n    groups_path = out / "groups.csv"\n    if groups_path.exists():\n        with groups_path.open(encoding="utf-8-sig", newline="") as stream:\n            existing = list(csv.DictReader(stream))\n        known = {r["image"] for r in existing}\n        with groups_path.open("a", newline="", encoding="utf-8") as stream:\n            writer = csv.DictWriter(stream, fieldnames=["image", "group_id"])\n            writer.writerows(\n                {"image": r["image"], "group_id": ""}\n                for r in rows\n                if r["cccd_candidate"] and r["image"] not in known\n            )\n    else:\n        with groups_path.open("w", newline="", encoding="utf-8") as stream:\n            writer = csv.DictWriter(stream, fieldnames=["image", "group_id"])\n            writer.writeheader()\n            writer.writerows(\n                {"image": r["image"], "group_id": ""} for r in rows if r["cccd_candidate"]\n            )\n    return report, rows\n\n\ndef grouped_split(rows, groups_path, seed=42):\n    """Split supplied groups; callers must record whether they are verified."""\n    with Path(groups_path).open(encoding="utf-8-sig", newline="") as stream:\n        records = list(csv.DictReader(stream))\n    groups = {r["image"]: r["group_id"].strip() for r in records}\n    if len(groups) != len(records):\n        raise ValueError("Duplicate image rows in groups.csv")\n    if any(not groups.get(r["image"]) for r in rows):\n        raise ValueError(\n            "Fill group_id for every CCCD candidate; same person = same anonymous group_id"\n        )\n    duplicates = defaultdict(set)\n    for row in rows:\n        duplicates[row["pixel_sha256"]].add(groups[row["image"]])\n    if any(len(v) > 1 for v in duplicates.values()):\n        raise ValueError("Identical images assigned different identities")\n    ids = sorted(\n        {groups[r["image"]] for r in rows},\n        key=lambda g: hashlib.sha256(f"{seed}:{g}".encode()).hexdigest(),\n    )\n    if len(ids) < 10:\n        raise ValueError("At least 10 groups required; inspect auto_group_report.json or manual groups")\n    n_val = max(1, round(len(ids) * 0.15))\n    allocation = {\n        g: ("valid" if i < n_val else "test" if i < 2 * n_val else "train")\n        for i, g in enumerate(ids)\n    }\n    return [\n        {**r, "split": allocation[groups[r["image"]]], "group_id": groups[r["image"]]} for r in rows\n    ]\n\n\ndef export_crops(root, rows, out, limit=80):\n    """Create an annotation queue, never pseudo ground truth; train images only."""\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    csv_path = out / "transcriptions.csv"\n    if csv_path.exists():\n        return csv_path\n    records = []\n    for row in rows:\n        if row["split"] != "train" or not row["cccd_candidate"]:\n            continue\n        with Image.open(Path(root) / row["image"]) as image:\n            image = image.convert("RGB")\n            for i, box in enumerate(row["boxes"]):\n                if box["field"] not in FIELDS:\n                    continue\n                filename = f"{row[\'sha256\'][:20]}_{i}_{box[\'field\']}.png"\n                x1, y1, x2, y2 = box["xyxy"]\n                image.crop((math.floor(x1), math.floor(y1), math.ceil(x2), math.ceil(y2))).save(\n                    out / filename\n                )\n                records.append(\n                    {\n                        "crop": filename,\n                        "image": row["image"],\n                        "field": box["field"],\n                        "text": "",\n                        "reviewed": "false",\n                    }\n                )\n                if len(records) >= limit:\n                    break\n        if len(records) >= limit:\n            break\n    with csv_path.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=["crop", "image", "field", "text", "reviewed"])\n        writer.writeheader()\n        writer.writerows(records)\n    return csv_path\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("root", type=Path)\n    parser.add_argument("--out", type=Path, required=True)\n    args = parser.parse_args()\n    report, rows = audit(args.root, args.out)\n    export_crops(args.root, rows, args.out / "recognition_queue")\n    print(\n        json.dumps(\n            {k: v for k, v in report.items() if not k.startswith("cross_")},\n            ensure_ascii=False,\n            indent=2,\n        )\n    )\n',
    encoding="utf-8",
)

In [ ]:
Path("/content/train_card_detector.py").write_text(
    '"""Colab detector baseline. Metrics are field-box metrics, not OCR accuracy."""\n\nimport hashlib\nimport json\nimport random\nfrom pathlib import Path\n\nimport torch\nimport torchvision\nfrom card_dataset import FIELDS, sha256\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn\nfrom torchvision.models.detection.faster_rcnn import FastRCNNPredictor\nfrom torchvision.ops import box_iou\nfrom torchvision.transforms.functional import to_tensor\n\n\nclass Cards(Dataset):\n    def __init__(self, root, rows):\n        self.root, self.rows = Path(root), rows\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        with Image.open(self.root / row["image"]) as image:\n            tensor = to_tensor(image.convert("RGB"))\n        annotations = [b for b in row["boxes"] if b["field"] in FIELDS]\n        target = {\n            "boxes": torch.tensor([b["xyxy"] for b in annotations], dtype=torch.float32).reshape(\n                -1, 4\n            ),\n            "labels": torch.tensor(\n                [FIELDS.index(b["field"]) + 1 for b in annotations], dtype=torch.int64\n            ),\n        }\n        return tensor, target\n\n\ndef build_model(pretrained=True, image_size=640):\n    model = fasterrcnn_mobilenet_v3_large_320_fpn(\n        weights="DEFAULT" if pretrained else None,\n        weights_backbone=None,\n        min_size=image_size,\n        max_size=image_size,\n    )\n    features = model.roi_heads.box_predictor.cls_score.in_features\n    model.roi_heads.box_predictor = FastRCNNPredictor(features, len(FIELDS) + 1)\n    return model\n\n\ndef collate(batch):\n    return tuple(zip(*batch, strict=True))\n\n\n@torch.inference_mode()\ndef evaluate(model, loader, device):\n    """One-to-one matching, IoU>=.5 and score>=.5; no threshold tuning."""\n    model.eval()\n    stats = {name: {"tp": 0, "fp": 0, "fn": 0} for name in FIELDS}\n    for images, targets in loader:\n        predictions = model([image.to(device) for image in images])\n        for pred, target in zip(predictions, targets, strict=True):\n            pred = {k: v.cpu() for k, v in pred.items()}\n            for category, name in enumerate(FIELDS, 1):\n                keep = (pred["labels"] == category) & (pred["scores"] >= 0.5)\n                predicted = pred["boxes"][keep]\n                predicted = predicted[pred["scores"][keep].argsort(descending=True)]\n                truth = target["boxes"][target["labels"] == category]\n                overlaps = box_iou(predicted, truth)\n                used = set()\n                for scores in overlaps:\n                    available = [(float(s), j) for j, s in enumerate(scores) if j not in used]\n                    score, j = max(available, default=(0.0, -1))\n                    if score >= 0.5:\n                        used.add(j)\n                        stats[name]["tp"] += 1\n                    else:\n                        stats[name]["fp"] += 1\n                stats[name]["fn"] += len(truth) - len(used)\n    tp, fp, fn = (sum(s[key] for s in stats.values()) for key in ("tp", "fp", "fn"))\n    return {\n        "precision": tp / max(1, tp + fp),\n        "recall": tp / max(1, tp + fn),\n        "f1": 2 * tp / max(1, 2 * tp + fp + fn),\n        "per_class_counts": stats,\n        "iou": 0.5,\n        "score": 0.5,\n        "metric": "micro field-box F1 (not mAP or OCR accuracy)",\n    }\n\n\ndef save_checkpoint(path, payload):\n    path = Path(path)\n    temporary = path.with_suffix(".tmp")\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\n\ndef checkpoint_config_diff(old_config, config):\n    """Accept absent legacy grouping metadata only for the identical manifest."""\n    old_config = dict(old_config)\n    if (\n        "grouping" not in old_config\n        and config.get("manifest_sha256")\n        and old_config.get("manifest_sha256") == config["manifest_sha256"]\n    ):\n        old_config["grouping"] = config.get("grouping")\n    return {\n        key: {"checkpoint": old_config.get(key), "current": config.get(key)}\n        for key in sorted(old_config.keys() | config.keys())\n        if old_config.get(key) != config.get(key)\n    }\n\n\ndef train(\n    root,\n    rows,\n    out,\n    smoke=True,\n    epochs=15,\n    batch_size=2,\n    resume=True,\n    allow_cpu=False,\n    pretrained=True,\n):\n    if not torch.cuda.is_available() and not allow_cpu:\n        raise RuntimeError("Colab: Runtime > Change runtime type > T4 GPU")\n    if not smoke and any(not r.get("group_id") for r in rows):\n        raise ValueError("Full training requires group IDs; use manual or provisional auto grouping")\n    random.seed(42)\n    torch.manual_seed(42)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    train_rows = [r for r in rows if r["split"] == "train"]\n    valid_rows = [r for r in rows if r["split"] == "valid"]\n    if smoke:\n        train_rows, valid_rows, epochs = train_rows[:8], valid_rows[:4], 1\n    if not train_rows or not valid_rows:\n        raise ValueError("Empty train/validation split")\n    loaders = [\n        DataLoader(\n            Cards(root, subset),\n            batch_size=batch_size,\n            shuffle=is_train,\n            num_workers=0,\n            collate_fn=collate,\n        )\n        for subset, is_train in ((train_rows, True), (valid_rows, False))\n    ]\n    fingerprint = hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()\n    config = {\n        "smoke": smoke,\n        "manifest_sha256": fingerprint,\n        "fields": FIELDS,\n        "image_size": 640,\n        "lr": 0.0001,\n        "batch_size": batch_size,\n        "torch": str(torch.__version__),\n        "torchvision": str(torchvision.__version__),\n        "pretrained": pretrained,\n        "seed": 42,\n        "grouping": "provisional_ocr" if any(r.get("group_source") for r in rows) else "provided_groups",\n    }\n    last = out / "last.pt"\n    checkpoint = (\n        torch.load(last, map_location="cpu", weights_only=True)\n        if resume and last.exists()\n        else None\n    )\n    differences = checkpoint_config_diff(checkpoint["config"], config) if checkpoint else {}\n    if differences:\n        raise ValueError(\n            f"Checkpoint incompatible: {last}\\n"\n            + json.dumps(differences, ensure_ascii=False, indent=2)\n            + "\\nĐể tiếp tục run cũ, khôi phục đúng cấu hình/dữ liệu/môi trường. "\n            "Để train mới, đổi RUN_NAME và tính lại RUN = BASE / \'runs\' / RUN_NAME. "\n            "Giữ checkpoint cũ; RESUME=False không ghi đè run đã tồn tại."\n        )\n    if not resume and last.exists():\n        raise ValueError("Run already exists; choose a new RUN_NAME")\n    model = build_model(pretrained=pretrained and checkpoint is None).to(device)\n    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=0.0001)\n    start, best, history = 0, -1.0, []\n    if checkpoint:\n        model.load_state_dict(checkpoint["model"])\n        optimizer.load_state_dict(checkpoint["optimizer"])\n        torch.set_rng_state(checkpoint["rng"])\n        if device.type == "cuda" and checkpoint.get("cuda_rng"):\n            torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])\n        start, best, history = checkpoint["epoch"] + 1, checkpoint["best"], checkpoint["history"]\n    (out / "manifest.json").write_text(\n        json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    for epoch in range(start, epochs):\n        model.train()\n        loss_sum = 0.0\n        for images, targets in loaders[0]:\n            images = [image.to(device) for image in images]\n            targets = [{k: v.to(device) for k, v in target.items()} for target in targets]\n            optimizer.zero_grad(set_to_none=True)\n            loss = sum(model(images, targets).values())\n            if not torch.isfinite(loss):\n                raise RuntimeError("Nonfinite loss: stop and inspect data")\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)\n            optimizer.step()\n            loss_sum += float(loss.detach())\n        metrics = evaluate(model, loaders[1], device)\n        history.append(\n            {"epoch": epoch + 1, "train_loss": loss_sum / len(loaders[0]), "validation": metrics}\n        )\n        improved = metrics["f1"] > best\n        best = max(best, metrics["f1"])\n        payload = {\n            "model": model.state_dict(),\n            "optimizer": optimizer.state_dict(),\n            "epoch": epoch,\n            "best": best,\n            "history": history,\n            "config": config,\n            "rng": torch.get_rng_state(),\n            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else [],\n        }\n        if improved:\n            save_checkpoint(out / "best.pt", payload)\n        save_checkpoint(last, payload)\n        (out / "metrics.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n        print(\n            f"Epoch {epoch + 1}/{epochs}: loss={history[-1][\'train_loss\']:.4f}, val field-box F1={metrics[\'f1\']:.4f}",\n            flush=True,\n        )\n    (out / "checkpoint_sha256.txt").write_text(sha256(out / "best.pt"), encoding="utf-8")\n    return history\n',
    encoding="utf-8",
)

In [ ]:
Path("/content/auto_card_groups.py").write_text(
    '"""Provisional OCR grouping for experiments; never verified identity labels."""\n\nimport csv\nimport hashlib\nimport hmac\nimport json\nimport math\nimport re\nimport secrets\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageOps\n\nVERSION = "easyocr-id-v1"\n\n\ndef accepted_id(readings):\n    """Require both views to agree on twelve digits, without character fixes."""\n    if len(readings) != 2:\n        return None\n    texts = [re.sub(r"\\s+", "", text) for text, _ in readings]\n    if (\n        texts[0] == texts[1]\n        and re.fullmatch(r"[0-9]{12}", texts[0])\n        and len(set(texts[0])) > 1\n        and all(score >= 0.85 for _, score in readings)\n    ):\n        return texts[0]\n    return None\n\n\ndef read_id(reader, root, row):\n    boxes = [box for box in row["boxes"] if box["field"] == "id"]\n    if len(boxes) != 1:\n        return None\n    x1, y1, x2, y2 = boxes[0]["xyxy"]\n    with Image.open(Path(root) / row["image"]) as image:\n        crop = image.convert("RGB").crop(\n            (math.floor(x1), math.floor(y1), math.ceil(x2), math.ceil(y2))\n        )\n    if min(crop.size) < 2:\n        return None\n    crop = crop.resize((max(1, round(crop.width * 64 / crop.height)), 64), Image.Resampling.LANCZOS)\n    readings = []\n    for variant in (\n        crop,\n        ImageOps.expand(ImageOps.autocontrast(crop.convert("L")), border=8, fill=255),\n    ):\n        result = reader.recognize(np.asarray(variant), detail=1, paragraph=False)\n        if len(result) != 1:\n            return None\n        readings.append((result[0][1], float(result[0][2])))\n    return accepted_id(readings)\n\n\ndef resolve(rows, tokens):\n    """Propagate exact-pixel matches only; quarantine conflicting duplicates."""\n    pixels = defaultdict(list)\n    for row in rows:\n        pixels[row["pixel_sha256"]].append(row)\n    assigned, excluded = [], []\n    for members in pixels.values():\n        values = {tokens[r["image"]] for r in members if tokens.get(r["image"])}\n        for row in members:\n            if len(values) == 1:\n                assigned.append(\n                    {\n                        **row,\n                        "group_id": next(iter(values)),\n                        "group_source": VERSION,\n                        "person_split_verified": False,\n                    }\n                )\n            else:\n                excluded.append(\n                    {\n                        "image": row["image"],\n                        "reason": "conflicting_ids" if values else "id_unreadable_or_uncertain",\n                    }\n                )\n    return sorted(assigned, key=lambda r: r["image"]), excluded\n\n\ndef auto_group(root, rows, out, reader):\n    """Cache per-image evidence and export a separate CSV, preserving manual work."""\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    key_path = out / "grouping.key"\n    if not key_path.exists():\n        key_path.write_bytes(secrets.token_bytes(32))\n    key = key_path.read_bytes()\n    cache_path = out / "auto_group_cache.json"\n    cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}\n    tokens = {}\n    for i, row in enumerate(rows):\n        cache_key = hashlib.sha256(\n            json.dumps(\n                [VERSION, row["sha256"], row["label_sha256"], hashlib.sha256(key).hexdigest()]\n            ).encode()\n        ).hexdigest()\n        if cache_key not in cache:\n            number = read_id(reader, root, row)\n            token = (\n                "auto_" + hmac.new(key, number.encode(), hashlib.sha256).hexdigest()\n                if number\n                else None\n            )\n            cache[cache_key] = token\n            temporary = cache_path.with_suffix(".tmp")\n            temporary.write_text(json.dumps(cache), encoding="utf-8")\n            temporary.replace(cache_path)\n        tokens[row["image"]] = cache[cache_key]\n        if (i + 1) % 25 == 0 or i + 1 == len(rows):\n            print(f"Auto grouping: {i + 1}/{len(rows)}", flush=True)\n    accepted, excluded = resolve(rows, tokens)\n    destination = out / "groups_auto.csv"\n    with destination.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=["image", "group_id"])\n        writer.writeheader()\n        writer.writerows({k: row[k] for k in writer.fieldnames} for row in accepted)\n    report = {\n        "method": VERSION,\n        "total": len(rows),\n        "included": len(accepted),\n        "excluded": len(excluded),\n        "groups": len({r["group_id"] for r in accepted}),\n        "person_split_verified": False,\n        "limitation": "Repeated same-model OCR is not independent verification. Metrics describe only the automatically readable subset; identity leakage remains possible.",\n    }\n    (out / "auto_group_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    (out / "auto_group_excluded.json").write_text(json.dumps(excluded, indent=2), encoding="utf-8")\n    print(json.dumps(report, indent=2))\n    return accepted, destination\n',
    encoding="utf-8",
)

In [ ]:
import importlib
import sys

sys.path.insert(0, "/content")
import card_dataset
import train_card_detector

importlib.reload(card_dataset)
importlib.reload(train_card_detector)

## Tải bản v1, kiểm SHA-256 và giải nén an toàn

In [ ]:
import urllib.request
import zipfile

from card_dataset import sha256

ARCHIVE = BASE / "vietnamese_card_v1.zip"
EXPECTED = "72d1b4883dd07a87f134a77123987d9b4107745ed2197c8cfb2d72643510d7db"
URL = "https://www.kaggle.com/api/v1/datasets/download/tranphucthaitruong/vietnamese-card-dataset?datasetVersionNumber=1"
if not ARCHIVE.exists():
    partial = ARCHIVE.with_suffix(".part")
    with urllib.request.urlopen(URL, timeout=120) as response, partial.open("wb") as target:
        import shutil

        shutil.copyfileobj(response, target)
    assert sha256(partial) == EXPECTED, "Archive khác bản đã kiểm tra; dừng để xác minh nguồn."
    partial.replace(ARCHIVE)
assert sha256(ARCHIVE) == EXPECTED, "Sai checksum dataset; không tiếp tục train."
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as archive:
    assert sum(item.file_size for item in archive.infolist()) < 500_000_000
    for item in archive.infolist():
        assert (ROOT / item.filename).resolve().is_relative_to(ROOT.resolve()), "Unsafe zip path"
    archive.extractall(ROOT)
print("Dataset SHA-256:", EXPECTED)

In [ ]:
import json

from card_dataset import audit, export_crops, grouped_split

report, rows = audit(ROOT, AUDIT)
print(
    json.dumps({k: report[k] for k in ["images", "splits", "cccd_candidates", "sizes"]}, indent=2)
)
print("Lỗi nhãn/ảnh:", len(report["errors"]))
print("Nhóm pixel trùng xuyên tập:", len(report["cross_split_identical_pixels"]))
print("Nhóm tên nguồn cần xem:", len(report["cross_split_source_name_candidates"]))
assert not report["errors"], "Xem audit.json và sửa lỗi dữ liệu trước khi train."
selected = [row for row in rows if row["cccd_candidate"]]
if not SMOKE:
    if AUTO_GROUP:
        import auto_card_groups
        import easyocr

        importlib.reload(auto_card_groups)
        reader = easyocr.Reader(
            ["en"],
            gpu=torch.cuda.is_available(),
            detector=False,
            model_storage_directory=str(BASE / "easyocr_models"),
        )
        selected, auto_csv = auto_card_groups.auto_group(ROOT, selected, AUDIT, reader)
        selected = grouped_split(selected, auto_csv)
        del reader
        torch.cuda.empty_cache()
        print(
            "AUTO: nhóm tạm bằng OCR, chưa xác minh theo người; không dùng làm benchmark chính thức."
        )
    else:
        selected = grouped_split(selected, GROUPS)
export_crops(ROOT, selected, RUN / "recognition_queue", limit=80)
print("Nhóm thủ công giữ nguyên:", GROUPS)

## Kiểm tra ảnh/box trước khi train
Các ảnh dưới đây chỉ từ train. Xem cả `cccd` có thực sự chọn đúng loại thẻ,
box có cắt mất dấu/ký tự, và các dòng địa chỉ có đủ không.
Để chạy tự động: SMOKE=False, AUTO_GROUP=True. Đọc 2 biến thể crop số CCCD,
chỉ nhận khi cùng 12 chữ số với confidence >=0.85; ảnh mâu thuẫn/không chắc bị loại.
Không sửa ký tự hay ép chỉ nhận số. Ngưỡng này là heuristic, chưa hiệu chỉnh.
Lưu groups_auto.csv, auto_group_report.json và auto_group_excluded.json trong audit_v1.
Chạy lại tận dụng cache, không ghi đè nhóm thủ công. Cần ít nhất 10 nhóm tự nhận được.
AUTO_GROUP=False vẫn dùng groups.csv đã điền. Tách nhóm 70/15/15 xấp xỉ.
Hash/pixel trùng chỉ tìm được một phần rò rỉ; cần người kiểm tra ảnh khác của cùng một thẻ.

In [ ]:
from card_dataset import FIELDS
from IPython.display import display
from PIL import Image, ImageDraw

for row in [r for r in selected if r["split"] == "train"][:3]:
    with Image.open(ROOT / row["image"]) as source:
        preview = source.convert("RGB")
    draw = ImageDraw.Draw(preview)
    for box in row["boxes"]:
        if box["field"] in FIELDS:
            draw.rectangle(box["xyxy"], outline="red", width=2)
            draw.text(tuple(box["xyxy"][:2]), box["field"], fill="red")
    display(preview)

## Fine-tune và tiếp tục khi mất phiên
`last.pt` lưu model, optimizer, RNG, epoch; `best.pt` chọn bằng **validation field-box F1**
với IoU≥0.5 và score≥0.5 cố định. Đây không phải mAP, cũng không phải độ chính xác OCR.
Mất phiên: chạy lại các cell, giữ RUN_NAME, SMOKE và cấu hình, RESUME=True.
Đổi môi trường/data/batch size: tạo RUN_NAME mới. Không nạp checkpoint không rõ nguồn.

In [ ]:
from train_card_detector import train

history = train(
    ROOT, selected, RUN, smoke=SMOKE, epochs=EPOCHS, batch_size=BATCH_SIZE, resume=RESUME
)
print("Checkpoint và metrics:", RUN)
print(json.dumps(history[-1], indent=2))

## Dùng kết quả
Giữ `best.pt`, `manifest.json`, `metrics.json`, `checkpoint_sha256.txt` trên Drive.
Chạy smoke thành công chỉ chứng minh pipeline chạy; không đưa số smoke vào báo cáo.
Chưa tự thay detector đang chạy trong identity_lab: cần đánh giá crop và OCR end-to-end trước.

Muốn fine-tune **VietOCR**: điền `text` chính xác và `reviewed=true` trong
`runs/<RUN_NAME>/recognition_queue/transcriptions.csv`, mở ảnh crop để kiểm từng dấu. Đây chỉ là hàng đợi 80 crop
đầu tiên, chưa phải tập train đủ và chưa có validation transcript.
Không coi dự đoán OCR là ground truth. Giữ split theo người khi mở rộng bộ recognition.

Tham khảo API model: https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_mobilenet_v3_large_320_fpn.html